# AutoShop Multi-Agent Orchestrator mit OpenAI Agents SDK

Dieses Notebook zeigt eine **echte Manager-/Orchestrator-Multi-Agent-Architektur**
mit dem OpenAI Agents SDK.

Im Unterschied zur Graph-Version wird der Ablauf **nicht durch einen selbstgebauten
Python-State-Graph** gesteuert. Stattdessen entscheidet ein zentraler
`AutoShop Manager Agent`, welche spezialisierten Agenten für die aktuelle Anfrage
benötigt werden.

Die Spezialisten werden mit `Agent.as_tool()` als Tools des Managers exponiert.

```text
                         User
                          |
                          v
                 +-------------------+
                 | AutoShop Manager  |
                 |   Orchestrator    |
                 +---------+---------+
                           |
            +--------------+--------------+
            |              |              |
            v              v              v
    +---------------+ +------------+ +------------+
    | AutoShop      | | Research   | | Offer      |
    | Specialist    | | Specialist | | Specialist |
    |               | |            | |            |
    | MCP Tools     | | kein MCP   | | kein MCP   |
    +-------+-------+ +------------+ +------------+
            |
            v
      AutoShop MCP
```

**Pattern:** Orchestrator–Workers / Manager with Agents as Tools

Der Manager bleibt während des gesamten Runs für die Benutzerantwort verantwortlich.
Die Spezialagenten bearbeiten klar abgegrenzte Teilaufgaben und geben ihre Resultate
an den Manager zurück.


---

## Abgrenzung zu den anderen Notebooks

| Notebook | Steuerung |
|---|---|
| `30-autoshop-agent.ipynb` | manueller Responses-API-/Tool-Loop |
| `31-autoshop-agent-sdk.ipynb` | Agents SDK, sequenziell aus Python aufgerufen |
| `32-autoshop-agent-graph.ipynb` | expliziter Python-State-Graph mit Conditional Routing |
| `33-autoshop-agent-orchestrator.ipynb` | **Manager-Agent entscheidet über Spezialagenten** |

Die Orchestrator-Version ist damit die reine Multi-Agent-Variante ohne eigene
Workflow-Engine.


---

## Installation


In [ ]:
%pip install -q -U openai openai-agents "mcp>=1.19,<3"

## Umgebung und Verbindung

Verwendet wird weiterhin `~/data/env.py`.

Der MCP-Endpunkt wird wie in den bisherigen AutoShop-Notebooks aus Kubernetes ermittelt.


In [ ]:
%run ~/data/env.py

import subprocess

from openai import AsyncOpenAI

from agents import (
    Agent,
    Runner,
    set_default_openai_client,
    set_tracing_disabled,
)

from agents.mcp import (
    MCPServerStreamableHttp,
    create_static_tool_filter,
)


def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True,
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True,
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"


MCP_URL = get_server_url()

openai_client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

set_default_openai_client(
    openai_client,
    use_for_tracing=False,
)

# Bei OpenAI-kompatiblen Endpoints ausserhalb der OpenAI Platform
# ist OpenAI-Tracing nicht zwingend verfügbar.
set_tracing_disabled(True)

print(f"MCP URL: {MCP_URL}")
print(f"Model:   {AI_MODEL}")


---

## Read-only MCP-Toolset

Nur lesende Tools werden dem AutoShop-Spezialisten zur Verfügung gestellt.
Destruktive Operationen wie `order_delete` bleiben ausgeschlossen.


In [ ]:
READ_ONLY_TOOLS = [
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
]

read_only_filter = create_static_tool_filter(
    allowed_tool_names=READ_ONLY_TOOLS,
)


## MCP-Server

Nur der interne AutoShop-Spezialist erhält Zugriff auf diesen MCP-Server.
Der Manager selbst hat keinen direkten Zugriff auf die internen Shop-Tools.


In [ ]:
autoshop_mcp = MCPServerStreamableHttp(
    name="AutoShop MCP",
    params={
        "url": MCP_URL,
        "timeout": 30,
    },
    cache_tools_list=True,
    tool_filter=read_only_filter,
    max_retry_attempts=2,
)


---

# 1. AutoShop Specialist

Dieser Agent ist der einzige Agent mit Zugriff auf interne AutoShop-Daten.

Seine Aufgabe ist bewusst eng definiert:

- Fahrzeuge im Catalog suchen
- interne Kunden- und Auftragsdaten lesen
- nur tatsächlich vorhandene Daten verwenden
- keine Preise, IDs oder Verfügbarkeiten erfinden


In [ ]:
autoshop_agent = Agent(
    name="AutoShop Specialist",
    model=AI_MODEL,
    instructions="""
Du bist der interne AutoShop-Spezialist.

Deine Aufgabe:
- Beschaffe interne AutoShop-Daten ausschliesslich über die verfügbaren MCP-Tools.
- Suche passende Fahrzeuge im Catalog.
- Lies Kunden- oder Auftragsdaten, wenn die Aufgabe dies verlangt.
- Verwende Preise, IDs, Fahrzeugdaten und Verfügbarkeiten ausschliesslich aus MCP.
- Wähle bei einer Fahrzeugempfehlung die passendsten verfügbaren Fahrzeuge aus.
- Begründe deine Auswahl kurz und sachlich.

Regeln:
- Erfinde keine internen Daten.
- Erfinde keine Preise.
- Erfinde keine Fahrzeug-IDs.
- Erfinde keine Verfügbarkeiten.
- Antworte auf Deutsch.
""",
    mcp_servers=[autoshop_mcp],
)


---

# 2. Research Specialist

Dieser Agent hat **keinen MCP-Zugriff**.

Er darf nur allgemeine Zusatzinformationen liefern, zum Beispiel:

- typische Vorteile eines Fahrzeugtyps
- sinnvolle Extras
- allgemeine Kaufkriterien
- Verkaufsargumente

Interne AutoShop-Fakten dürfen nicht erfunden werden.


In [ ]:
research_agent = Agent(
    name="Vehicle Research Specialist",
    model=AI_MODEL,
    instructions="""
Du bist ein externer Fahrzeug-Research-Spezialist ohne Zugriff auf den AutoShop.

Du darfst:
- allgemeine Fahrzeuginformationen ergänzen,
- typische Vorteile eines Fahrzeugtyps erklären,
- sinnvolle Extras als unverbindliche Vorschläge nennen,
- allgemeine Kauf- und Nutzungskriterien erläutern.

Du darfst nicht:
- AutoShop-Preise erfinden,
- AutoShop-Verfügbarkeiten erfinden,
- interne IDs erfinden,
- konkrete interne Shop-Daten behaupten.

Kennzeichne Zusatzinformationen als allgemein oder unverbindlich,
wenn sie nicht aus dem AutoShop stammen.

Antworte auf Deutsch.
""",
)


---

# 3. Offer Specialist

Dieser Agent erstellt aus den vom Manager gelieferten Informationen eine strukturierte
Demo-Offerte.

Auch dieser Agent hat keinen eigenen Zugriff auf MCP.


In [ ]:
offer_agent = Agent(
    name="Offer Specialist",
    model=AI_MODEL,
    instructions="""
Du bist der AutoShop-Offert-Spezialist.

Erstelle aus den übergebenen Informationen eine übersichtliche Markdown-Offerte.

Regeln:
- Fahrzeugdaten, IDs und Preise dürfen nur aus den übergebenen internen AutoShop-Daten stammen.
- Erfinde keine Preise, IDs, Fahrzeuge oder Verfügbarkeiten.
- Externe Zusatzinformationen dürfen nur als unverbindliche Empfehlungen verwendet werden.
- Die Offerte muss klar zwischen Shop-Daten und allgemeinen Zusatzinformationen unterscheiden.
- Füge den Hinweis hinzu:
  "Diese Offerte ist ein Education-Beispiel und nicht rechtsverbindlich."
- Antworte auf Deutsch.
""",
)


---

# 4. Spezialagenten als Tools

Hier findet der entscheidende Schritt der Multi-Agent-Orchestrierung statt.

Mit `Agent.as_tool()` werden komplette Agenten zu aufrufbaren Tools des Managers.

Der Manager kann dadurch selbst entscheiden:

```text
Benötige ich interne Daten?
    -> search_autoshop

Benötige ich allgemeines Zusatzwissen?
    -> research_vehicle

Soll eine Offerte erzeugt werden?
    -> create_offer
```

Der Spezialagent übernimmt dabei **nicht** die Benutzerkonversation.
Nach dem Tool-Aufruf geht die Kontrolle wieder an den Manager zurück.


In [ ]:
autoshop_tool = autoshop_agent.as_tool(
    tool_name="search_autoshop",
    tool_description=(
        "Liest interne AutoShop-Daten über MCP. "
        "Verwende dieses Tool für Catalog-Fahrzeuge, Preise, interne Kunden- "
        "und Auftragsdaten sowie konkrete AutoShop-Empfehlungen."
    ),
)

research_tool = research_agent.as_tool(
    tool_name="research_vehicle",
    tool_description=(
        "Liefert allgemeine Fahrzeug-Zusatzinformationen, typische Vorteile, "
        "Kaufkriterien und unverbindliche Extras. Hat keinen Zugriff auf AutoShop-Daten."
    ),
)

offer_tool = offer_agent.as_tool(
    tool_name="create_offer",
    tool_description=(
        "Erstellt aus bereits beschafften internen AutoShop-Daten und optionalen "
        "Zusatzinformationen eine strukturierte Demo-Offerte."
    ),
)


---

# 5. AutoShop Manager / Orchestrator

Der Manager ist der einzige Agent, der direkt mit dem Benutzer interagiert.

Er besitzt drei Spezialisten als Tools und entscheidet autonom, welche davon für eine
Anfrage benötigt werden.

Wichtig ist die Rollenverteilung:

```text
Manager
  |
  +-- plant
  +-- delegiert
  +-- kombiniert Resultate
  +-- formuliert finale Antwort

Worker
  |
  +-- lösen klar abgegrenzte Teilaufgaben
```


In [ ]:
manager_agent = Agent(
    name="AutoShop Manager",
    model=AI_MODEL,
    instructions="""
Du bist der zentrale AutoShop Manager und koordinierst spezialisierte Agenten.

Du bist für die finale Benutzerantwort verantwortlich.

Verfügbare Spezialisten:

1. search_autoshop
   - für interne AutoShop-Daten
   - Catalog-Fahrzeuge
   - interne Preise
   - Kunden- und Auftragsdaten

2. research_vehicle
   - für allgemeine externe Fahrzeug-Zusatzinformationen
   - typische Vorteile
   - sinnvolle Extras
   - allgemeine Kaufkriterien

3. create_offer
   - für die Erstellung einer strukturierten Demo-Offerte

Arbeitsweise:
- Analysiere zuerst die Benutzeranfrage.
- Rufe nur Spezialisten auf, die tatsächlich benötigt werden.
- Wenn konkrete AutoShop-Fahrzeuge, Preise, Kunden oder Aufträge relevant sind,
  musst du search_autoshop verwenden.
- Wenn eine Offerte verlangt wird, beschaffe zuerst die nötigen internen Daten
  und rufe anschliessend create_offer auf.
- Verwende research_vehicle nur, wenn allgemeine Zusatzinformationen einen
  fachlichen Mehrwert liefern.
- Gib dem Offer-Spezialisten alle relevanten bereits beschafften Informationen mit.
- Kombiniere die Resultate zu einer konsistenten finalen Antwort.
- Erfinde niemals AutoShop-Preise, IDs, Fahrzeuge oder Verfügbarkeiten.
- Antworte auf Deutsch.

Wichtig:
Der Manager soll die Arbeit koordinieren und nicht versuchen, interne
AutoShop-Daten aus eigenem Modellwissen zu beantworten.
""",
    tools=[
        autoshop_tool,
        research_tool,
        offer_tool,
    ],
)


---

# 6. Orchestrator ausführen

Der gesamte Multi-Agent-Workflow wird jetzt mit **einem einzigen `Runner.run()`**
gestartet.

Es gibt keine expliziten Python-Routing-Funktionen und keinen selbstgebauten Graph.
Der Manager entscheidet im Agent-Loop selbst über seine nächsten Tool-/Agent-Aufrufe.


In [ ]:
async def run_autoshop_orchestrator(
    customer_name: str,
    user_request: str,
):
    prompt = f"""
Kunde:
{customer_name}

Anfrage:
{user_request}
"""

    async with autoshop_mcp:
        result = await Runner.run(
            manager_agent,
            prompt,
            max_turns=20,
        )

    return result


---

# 7. Beispiel: Fahrzeugempfehlung mit Offerte

Bei dieser Anfrage wird der Manager typischerweise:

1. `search_autoshop` aufrufen,
2. optional `research_vehicle` verwenden,
3. `create_offer` aufrufen,
4. die Resultate zu einer finalen Antwort zusammenführen.

Die konkrete Reihenfolge wird jedoch nicht von Python vorgegeben, sondern vom Manager-Agent.


In [ ]:
from IPython.display import Markdown, display

result = await run_autoshop_orchestrator(
    customer_name="Max Muster",
    user_request=(
        "Ich suche ein günstiges Auto mit viel Platz für die Familie. "
        "Bitte empfehle mir ein passendes Fahrzeug und erstelle eine Offerte."
    ),
)

display(Markdown(result.final_output))


---

# 8. Run-Items und Agentenaktivität anzeigen

Das Agents SDK liefert neben `final_output` auch die während des Runs erzeugten Items.

Damit kann nachvollzogen werden, welche Tools bzw. Spezialagenten der Manager verwendet hat.


In [ ]:
print(f"Finaler Agent: {result.last_agent.name}")
print(f"Anzahl Run-Items: {len(result.new_items)}")
print()

for i, item in enumerate(result.new_items, start=1):
    print(f"{i:02d}: {type(item).__name__}")


## Detaillierte Run-Items

Die genaue Struktur einzelner Items hängt vom jeweiligen SDK-Release und vom
ausgeführten Tool-Pfad ab. Für die Exploration im Notebook können die Item-Objekte
direkt ausgegeben werden.


In [ ]:
for i, item in enumerate(result.new_items, start=1):
    print("=" * 80)
    print(f"ITEM {i}: {type(item).__name__}")
    print(item)


---

# 9. Beispiel ohne Offerte

Bei einer reinen Informationsanfrage soll der Manager den Offer-Spezialisten nicht
unnötig aufrufen.

Beispiel:

```python
info_result = await run_autoshop_orchestrator(
    customer_name="Max Muster",
    user_request="Welche Fahrzeuge im AutoShop eignen sich für eine Familie mit viel Gepäck?",
)
```

Der Manager entscheidet dabei selbst, ob zusätzliches Research einen Mehrwert bietet.


---

# 10. Unterschied zum Python-Graph

## Orchestrator–Workers

```text
                    Manager Agent
                         |
            +------------+------------+
            |            |            |
            v            v            v
         AutoShop     Research       Offer
          Agent        Agent         Agent
            |
            v
           MCP
```

Routing:

```text
LLM entscheidet -> welcher Worker wird aufgerufen?
```

## Graph-Version

```text
                    Python Workflow
                         |
              Conditional Edges
                         |
            +------------+------------+
            |            |            |
            v            v            v
          Lookup      Research       Offer
            |            |            |
            v            v            v
          Agent        Agent         Agent
```

Routing:

```text
Python/State entscheidet -> welcher Node wird ausgeführt?
```

Der Orchestrator ist flexibler und benötigt weniger Workflow-Code.
Der Graph bietet dafür mehr deterministische Kontrolle über Reihenfolge,
State, Retry-Pfade und langfristige Workflow-Ausführung.


---

# 11. Pattern

## Orchestrator–Workers Pattern

Ein zentraler Orchestrator analysiert eine Aufgabe und delegiert klar abgegrenzte
Teilaufgaben an spezialisierte Worker. Die Worker liefern ihre Ergebnisse an den
Orchestrator zurück, der daraus die finale Antwort erstellt.

In diesem Notebook:

```text
AutoShop Manager
    +
AutoShop Specialist
    +
Research Specialist
    +
Offer Specialist
    =
Orchestrator–Workers Pattern
```

Die Umsetzung erfolgt direkt mit dem OpenAI Agents SDK über `Agent.as_tool()`.


---

# 12. Wann diese Variante sinnvoll ist

Die Orchestrator-Version eignet sich besonders, wenn:

- die benötigte Reihenfolge nicht vollständig im Voraus bekannt ist,
- verschiedene Spezialisten je nach Anfrage benötigt werden,
- ein zentraler Agent die finale Antwort verantworten soll,
- mehrere Worker-Resultate zusammengeführt werden müssen,
- möglichst wenig eigene Workflow-Logik implementiert werden soll.

Ein expliziter Graph ist sinnvoller, wenn:

- feste Prozessschritte garantiert werden müssen,
- langlebiger persistenter State benötigt wird,
- Retry-, Timeout- oder Human-Approval-Pfade explizit modelliert werden sollen,
- regulatorische oder fachliche Vorgaben einen deterministischen Ablauf verlangen.


---

## Referenz

OpenAI Agents SDK:

- Agent orchestration: https://openai.github.io/openai-agents-python/multi_agent/
- Agents as tools: https://openai.github.io/openai-agents-python/tools/
- Agents: https://openai.github.io/openai-agents-python/agents/
